# Linly-Dubbing Kaggle WebUI (v4.1 - Technical Fixes Applied)

This notebook is optimized for running **Linly-Dubbing** on Kaggle with **Dual T4 GPUs**. Recent updates include fixes for submodule path conflicts and more robust dependency installation.

### 🚀 Key Improvements in v4.1
- **Submodule Isolation**: Fixed `sys.path` order to prevent submodules (like demucs) from shadowing main project files.
- **Robust Dependencies**: Improved bash syntax and error handling in dependency installation.
- **Resilient AI Model Downloads**: Added retry capability for Hugging Face snapshot downloads to handle server timeouts.
- **Python 3.12 Compatibility**: Enhanced system library installation for building audio tools from source.

### 📋 Execution Guide
1. **Step 1**: Clone repository and check GPU availability
2. **Step 2**: Install all dependencies (Uses robust multi-stage install)
3. **Step 3**: Download required AI models
4. **Step 4**: Launch the WebUI

### ⚙️ Kaggle Setup Requirements
- Enable **GPU T4 x2** in Settings → Accelerator
- Enable **Internet** in Settings → Internet

---

In [1]:
# ============================================================================
# Step 0: 环境检测 (Environment Detection)
# ============================================================================

import os
import sys
import platform
import subprocess
import shutil

print("=" * 60)
print("🔍 Kaggle 运行环境检测")
print("=" * 60)

# Python 信息
print("\n📊 Python 环境:")
print(f"   Python 版本: {sys.version.split()[0]}")
print(f"   Python 路径: {sys.executable}")

# CUDA 信息
print("\n🎮 CUDA 环境:")
try:
    import torch
    print(f"   PyTorch 版本: {torch.__version__}")
    print(f"   CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   CUDA 版本: {torch.version.cuda}")
        try:
            print(f"   cuDNN 版本: {torch.backends.cudnn.version()}")
        except: print("   cuDNN 版本: 获取失败")
except ImportError:
    print("   ⚠️ PyTorch 未安装")

# 磁盘空间
stat = shutil.disk_usage('/kaggle/working')
print(f"\n💾 磁盘可用空间: {stat.free / (1024**3):.1f} GB")

# 预装 Python 包 (Handle torchvision error gracefully)
print("\n📦 预装 Python 包 (关键):")
for pkg in ['torch', 'torchvision', 'numpy', 'pip']:
    try: 
        m = __import__(pkg)
        print(f"   ✅ {pkg}: {getattr(m, '__version__', 'OK')}")
    except Exception as e:
        print(f"   ⚠️ {pkg}: 导入失败 ({type(e).__name__})")

print("\n" + "=" * 60)
print("✅ 环境检测完成!")
print("=" * 60)

🔍 Kaggle 运行环境检测

📊 Python 环境:
   Python 版本: 3.12.12
   Python 路径: /usr/bin/python3

🎮 CUDA 环境:
   PyTorch 版本: 2.8.0+cu126
   CUDA 可用: True
   CUDA 版本: 12.6
   cuDNN 版本: 91002

💾 磁盘可用空间: 19.5 GB

📦 预装 Python 包 (关键):
   ✅ torch: 2.8.0+cu126
   ✅ torchvision: 0.23.0+cu126
   ✅ numpy: 2.0.2
   ✅ pip: 24.1.2

✅ 环境检测完成!


In [ ]:
# ============================================================================
# Step 1: Clone Repository and Check GPU
# ============================================================================

import os
import torch
import shutil

print("=" * 60)
print("🔍 GPU Detection")
print("=" * 60)
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"✅ Found {gpu_count} GPU(s):")
    for i in range(gpu_count):
        print(f"   - GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("❌ No GPU detected! Please enable GPU T4 x2 in settings.")

print("\n" + "=" * 60)
print("📦 Cloning Repository")
print("=" * 60)

# Always remove old directory and clone fresh to ensure latest changes
project_path = '/kaggle/working/Linly-Dubbing'
if os.path.exists(project_path):
    print(f"Removing existing directory: {project_path}")
    shutil.rmtree(project_path)

!cd /kaggle/working && git clone https://github.com/infinite-gaming-studio/Linly-Dubbing.git --depth 1

%cd /kaggle/working/Linly-Dubbing

print("\nInitializing submodules...")
!git submodule update --init --recursive

print("\n✅ Step 1 Complete!")

In [ ]:
# ============================================================================
# Step 2: Install Dependencies (Final Version - Numpy Compatibility Fix)
# ============================================================================

print("=" * 60)
print("📦 Installing System Dependencies")
print("=" * 60)

# 1. Install system tools required for build
!apt-get update -qq
!apt-get install -y -qq build-essential libfst-dev libfst-tools python3-dev ffmpeg \
    libavdevice-dev libavfilter-dev libavformat-dev libavcodec-dev \
    libswresample-dev libswscale-dev libavutil-dev

print("\n🐍 Installing Python Dependencies with Strict Version Control")
print("=" * 60)

# 2. CRITICAL: Install numpy and scipy FIRST with strict versions to prevent upgrades
print("Step 1/10: Installing numpy and scipy with strict versions...")
!pip install -q "numpy>=1.24.0,<2.0" "scipy>=1.10.0,<1.12.0"

# 3. Install other core scientific packages with compatible versions
print("Step 2/10: Installing core scientific packages...")
!pip install -q "pandas>=2.0,<2.3" "scikit-learn>=1.3.0"

# 4. Update pip tools
print("Step 3/10: Updating pip tools...")
!pip install --upgrade -q pip setuptools wheel

# 5. Install submodule base dependencies
print("Step 4/10: Installing submodule base dependencies...")
!pip install -q dora-search diffq einops julius lameenc tqdm treetable hydra-core hydra-colorlog pyyaml

# 6. Install TTS/audio dependencies with compatible versions
print("Step 5/10: Installing audio and ML dependencies...")
!pip install -q cython soundfile librosa inflect fsspec aiohttp packaging
!pip install -q "transformers>=4.39.0,<4.40.0" encodec unidecode num2words

# 7. Patch requirements.txt to avoid conflicts
print("Step 6/10: Patching requirements.txt...")
!sed -i 's/numpy<2.0.0/# numpy<2.0.0 # Already installed/g' requirements.txt
!sed -i 's/^scipy/# scipy # Already installed/g' requirements.txt
!sed -i 's/transformers==4.39.3/# transformers==4.39.3 # Already installed/g' requirements.txt

# 8. Patch TTS for Python 3.12 compatibility
if os.path.exists('submodules/TTS/setup.py'):
    !sed -i 's/Version(python_version) >= Version(\"3.12\"): /Version(python_version) >= Version(\"3.13\"): /g' submodules/TTS/setup.py
    !sed -i 's/python_requires=\">=3.9.0, <3.12\",/python_requires=\">=3.9.0, <3.13\",/g' submodules/TTS/setup.py

# 9. Optional: Try pynini (allow failure - it's not critical)
print("Step 7/10: Attempting pynini installation (optional)...")
!pip install -q pynini==2.1.5 --no-cache-dir 2>/dev/null || echo "⚠️  pynini installation skipped (non-critical)"

# 10. Install core requirements (numpy/scipy already installed)
print("Step 8/10: Installing core requirements...")
!pip install -q -r requirements.txt

# 11. Install whisperX dependencies
print("Step 9/10: Installing whisperX dependencies...")
!pip install -q pyannote.audio==3.1.1 faster-whisper==1.0.0

# 12. Install submodules with improved handling
print("Step 10/10: Installing submodules...")
submodules_info = [
    ('submodules/demucs', 'demucs'),
    ('submodules/whisper', 'openai-whisper'),
    ('submodules/whisperX', 'whisperX'),
    ('submodules/TTS', 'TTS')
]

for sm_path, sm_name in submodules_info:
    if os.path.exists(sm_path):
        print(f"   - Installing {sm_name}...")
        result = !pip install -q -e "{sm_path}" 2>&1
        if result and 'error' in str(result).lower():
            print(f"     Retrying {sm_name} without dependencies...")
            !pip install -q -e "{sm_path}" --no-deps || echo f"   ⚠️  {sm_name} install failed (will use sys.path fallback)"
        else:
            print(f"     ✓ {sm_name} installed successfully")

# 13. Final tools
print("\nInstalling final tools...")
!pip install -q loguru yt-dlp "gradio>=4.44,<5.0"

# 14. Verify critical package versions
print("\n" + "=" * 60)
print("🔍 Verifying Critical Package Versions")
print("=" * 60)
import sys
for pkg in ['numpy', 'scipy', 'transformers', 'pandas']:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f"   ✓ {pkg}: {ver}")
    except ImportError:
        print(f"   ✗ {pkg}: NOT INSTALLED")

print("\n" + "=" * 60)
print("✅ Step 2 Complete!")
print("=" * 60)

In [4]:
# ============================================================================
# Step 3: Download AI Models
# ============================================================================

print("=" * 60)
print("🤖 Downloading AI Models")
print("=" * 60)

!mkdir -p models/ASR/whisper
wav2vec_path = 'models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth'
if not os.path.exists(wav2vec_path):
    !wget -nc https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth -O {wav2vec_path}

# Use download script
!python scripts/huggingface_download.py

print("\n✅ Step 3 Complete!")

🤖 Downloading AI Models
--2026-01-30 02:27:19--  https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth
Resolving download.pytorch.org (download.pytorch.org)... 18.160.143.21, 18.160.143.101, 18.160.143.48, ...
Connecting to download.pytorch.org (download.pytorch.org)|18.160.143.21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 377664473 (360M) [application/x-www-form-urlencoded]
Saving to: ‘models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth’

models/ASR/whisper/ 100%[===================>] 360.17M   306MB/s    in 1.2s    

2026-01-30 02:27:21 (306 MB/s) - ‘models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth’ saved [377664473/377664473]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warning

In [ ]:
# ============================================================================
# Step 4: Launch WebUI
# ============================================================================

import os
import sys

print("=" * 60)
print("🚀 Launching Linly-Dubbing WebUI")
print("=" * 60)

# Ensure submodules are in path (Fallback for installation issues)
project_root = '/kaggle/working/Linly-Dubbing'
submodule_paths = [
    os.path.join(project_root, 'submodules/demucs'),
    os.path.join(project_root, 'submodules/whisper'),
    os.path.join(project_root, 'submodules/whisperX'),
    os.path.join(project_root, 'submodules/TTS')
]
for p in submodule_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.append(p)  # Use append to avoid shadowing project tools

# Set environment variables
os.environ['PYTHONPATH'] = ':'.join(submodule_paths) + (':' + os.environ.get('PYTHONPATH', '') if os.environ.get('PYTHONPATH') else '')
os.environ['MPLBACKEND'] = 'Agg'

if not os.path.exists('.env'):
    !cp env.example .env

# Verify critical imports before launch
print("\n🔍 Verifying critical imports...")
try:
    from tools.step000_video_downloader import download_from_url##
    from tools.step010_demucs_vr import separate_all_audio_under_folder
    print("✅ All critical modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("\n🔧 Attempting to fix missing dependencies...")
    # Install missing dora if needed
    !pip install -q dora-search
    print("Please re-run this cell after installing missing dependencies.")
    raise

print("\n🌐 Starting Gradio WebUI...")
!python webui.py